# 🎬 Netflix Movie Recommendation System
### Content-Based Filtering using TF-IDF & Cosine Similarity

---

**Project Overview:**
This notebook walks through a complete end-to-end machine learning project:
1. Dataset loading & inspection
2. Data cleaning
3. Exploratory Data Analysis (EDA) with visualizations
4. Feature Engineering (TF-IDF)
5. Building a Content-Based Recommendation System
6. Demo recommendations for sample movies

**Dataset:** [Netflix Movies and TV Shows – Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)

---
## 📦 Step 0: Import Libraries

In [ ]:
# ─────────────────────────────────────────────────────────────
# Import all required libraries
# ─────────────────────────────────────────────────────────────

import os
import sys
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for cleaner output

import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Machine Learning: TF-IDF and Cosine Similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity

# ── Global plot style ─────────────────────────────────────────
# Set a clean, consistent visual style for all charts
sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'font.family': 'DejaVu Sans'
})

print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'seaborn : {sns.__version__}')
print('All libraries imported successfully ✅')

---
## 📂 Step 1: Load the Dataset

> **Before running this cell**, make sure `netflix_titles.csv` is placed inside the `dataset/` folder.  
> See **README.md** for download instructions.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Load Dataset
# ─────────────────────────────────────────────────────────────

# Build the path relative to this notebook's location
NOTEBOOK_DIR = os.path.abspath('')          # Current working directory (notebook location)
DATASET_PATH = os.path.join(NOTEBOOK_DIR, 'dataset', 'netflix_titles.csv')

def load_dataset(path):
    """
    Load the Netflix CSV dataset. Provides a helpful error message
    if the file is not found.
    """
    if not os.path.exists(path):
        print('=' * 65)
        print('❌  DATASET NOT FOUND')
        print('=' * 65)
        print(f'Expected path: {path}')
        print()
        print('📥  How to get the dataset:')
        print()
        print('  Option A – Kaggle CLI:')
        print('    pip install kaggle')
        print('    kaggle datasets download -d shivamb/netflix-shows -p dataset/ --unzip')
        print()
        print('  Option B – Manual:')
        print('    Visit: https://www.kaggle.com/datasets/shivamb/netflix-shows')
        print('    Download and place netflix_titles.csv in the dataset/ folder.')
        print('=' * 65)
        raise FileNotFoundError(f'Dataset not found: {path}')
    
    df = pd.read_csv(path)
    print(f'✅  Dataset loaded successfully!')
    print(f'   Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
    return df

# Load the raw data
df_raw = load_dataset(DATASET_PATH)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Quick Preview of the Dataset
# ─────────────────────────────────────────────────────────────

print('─── First 5 Rows ───')
display(df_raw.head())

print('\n─── Dataset Info ───')
df_raw.info()

print('\n─── Statistical Summary ───')
display(df_raw.describe(include='all'))

In [ ]:
# ─────────────────────────────────────────────────────────────
# Check Missing Values
# ─────────────────────────────────────────────────────────────

missing = df_raw.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('─── Missing Values ───')
display(missing_df)

# Visualise missing values as a bar chart
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(missing_df.index, missing_df['Missing %'], color=sns.color_palette('husl', len(missing_df)))
    ax.set_title('Missing Values by Column (%)', pad=12)
    ax.set_xlabel('Column')
    ax.set_ylabel('Missing (%)')
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, missing_df['Missing %']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val}%',
                ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()

---
## 🧹 Step 2: Data Cleaning

In [ ]:
# ─────────────────────────────────────────────────────────────
# Data Cleaning Function
# ─────────────────────────────────────────────────────────────

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans the raw Netflix dataframe:
      1. Removes exact duplicate rows
      2. Fills NaN in text columns with empty string (needed for TF-IDF)
      3. Strips leading/trailing whitespace
      4. Converts release_year to numeric
    """
    df = df.copy()  # Never modify the original dataframe!
    
    # Step 2a: Remove duplicates
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    print(f'✂️  Removed {before - after} duplicate rows.')
    
    # Step 2b: Fill NaN in text columns used for recommendations
    text_cols = ['title', 'director', 'cast', 'listed_in', 'description', 'country', 'rating']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna('')  # Empty string instead of NaN
    
    # Step 2c: Strip whitespace from all string columns
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()
    
    # Step 2d: Convert release_year to numeric (handles any stray non-numeric values)
    if 'release_year' in df.columns:
        df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')
    
    # Reset index for clean sequential indexing
    df = df.reset_index(drop=True)
    
    print(f'✅  Cleaned dataset shape: {df.shape}')
    return df

# Apply cleaning
df = clean_data(df_raw)

# Confirm no NaN remains in key columns
print('\n─── Remaining NaN in text columns after cleaning ───')
display(df[['title','director','cast','listed_in','description']].isnull().sum())

---
## 📊 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# ─────────────────────────────────────────────────────────────
# EDA 1: Content Type Distribution (Movies vs TV Shows)
# ─────────────────────────────────────────────────────────────

type_counts = df['type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Pie Chart ---
colors = ['#E50914', '#564D4D']  # Netflix red and dark grey
wedges, texts, autotexts = axes[0].pie(
    type_counts.values,
    labels=type_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    explode=(0.05, 0),
    shadow=True
)
for text in autotexts:
    text.set_fontsize(13)
    text.set_color('white')
    text.set_fontweight('bold')
axes[0].set_title('Content Type: Movies vs TV Shows', pad=14)

# --- Bar Chart ---
bars = axes[1].bar(type_counts.index, type_counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[1].set_title('Count of Movies vs TV Shows', pad=14)
axes[1].set_ylabel('Count')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{int(bar.get_height()):,}', ha='center', va='bottom', fontweight='bold')

fig.suptitle('Netflix Content Type Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(type_counts.to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────
# EDA 2: Top 15 Genres Distribution
# ─────────────────────────────────────────────────────────────
# 'listed_in' contains multiple genres per row separated by commas.
# We split and explode them to count each genre individually.
# ─────────────────────────────────────────────────────────────

# Split the comma-separated genre string and explode into individual rows
all_genres = (
    df['listed_in']
    .str.split(', ')
    .explode()              # Each genre gets its own row
    .str.strip()
    .value_counts()
    .head(15)              # Top 15 genres only
)

fig, ax = plt.subplots(figsize=(13, 6))

# Use a gradient palette for visual appeal
palette = sns.color_palette('rocket_r', len(all_genres))
bars = ax.barh(all_genres.index[::-1], all_genres.values[::-1], color=palette[::-1], edgecolor='white')

# Add count labels on bars
for bar in bars:
    width = bar.get_width()
    ax.text(width + 10, bar.get_y() + bar.get_height()/2,
            f'{int(width):,}', va='center', ha='left', fontsize=9)

ax.set_title('Top 15 Most Common Genres on Netflix', pad=14)
ax.set_xlabel('Number of Titles')
ax.set_ylabel('Genre')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EDA 3: Release Year Distribution
# ─────────────────────────────────────────────────────────────

year_data = df[df['release_year'].notna()].copy()
year_data['release_year'] = year_data['release_year'].astype(int)

# Filter to a readable range (last 40 years dominate the catalogue)
year_data_filtered = year_data[year_data['release_year'] >= 1980]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# --- Overall histogram ---
axes[0].hist(year_data['release_year'], bins=50, color='#E50914', edgecolor='#1a1a1a', alpha=0.85)
axes[0].set_title('Distribution of Release Years – All Netflix Titles', pad=12)
axes[0].set_xlabel('Release Year')
axes[0].set_ylabel('Number of Titles')

# --- Stacked by type (Movies vs TV Shows), recent years ---
movies_by_year = year_data_filtered[year_data_filtered['type'] == 'Movie']['release_year'].value_counts().sort_index()
shows_by_year  = year_data_filtered[year_data_filtered['type'] == 'TV Show']['release_year'].value_counts().sort_index()

all_years = sorted(set(movies_by_year.index) | set(shows_by_year.index))
m_vals = [movies_by_year.get(y, 0) for y in all_years]
s_vals = [shows_by_year.get(y, 0) for y in all_years]

axes[1].bar(all_years, m_vals, label='Movie', color='#E50914', alpha=0.85)
axes[1].bar(all_years, s_vals, bottom=m_vals, label='TV Show', color='#564D4D', alpha=0.85)
axes[1].set_title('Movies vs TV Shows by Release Year (1980–Present)', pad=12)
axes[1].set_xlabel('Release Year')
axes[1].set_ylabel('Number of Titles')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EDA 4: Ratings Distribution
# ─────────────────────────────────────────────────────────────

# Filter out rows with missing or non-standard rating values
valid_ratings = ['G', 'PG', 'PG-13', 'R', 'NC-17', 'TV-Y', 'TV-Y7', 'TV-Y7-FV',
                 'TV-G', 'TV-PG', 'TV-14', 'TV-MA', 'NR', 'UR']
rating_data = df[df['rating'].isin(valid_ratings)]
rating_counts = rating_data['rating'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bar Chart ---
palette = sns.color_palette('viridis', len(rating_counts))
bars = axes[0].bar(rating_counts.index, rating_counts.values, color=palette, edgecolor='white', linewidth=0.8)
axes[0].set_title('Rating Distribution – All Content', pad=12)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=35)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)

# --- Ratings split by content type ---
rating_type = rating_data.groupby(['rating', 'type']).size().unstack(fill_value=0)
rating_type = rating_type.reindex(rating_counts.index)  # Keep same order
rating_type.plot(kind='bar', ax=axes[1], color=['#E50914', '#564D4D'], edgecolor='white', linewidth=0.8)
axes[1].set_title('Ratings Split: Movies vs TV Shows', pad=12)
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=35)
axes[1].legend(title='Content Type')

plt.suptitle('Netflix Content Ratings', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EDA 5: Top 15 Countries by Number of Titles
# ─────────────────────────────────────────────────────────────

all_countries = (
    df['country']
    .str.split(', ')
    .explode()
    .str.strip()
    .replace('', np.nan)
    .dropna()
    .value_counts()
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
palette = sns.color_palette('mako_r', len(all_countries))
bars = ax.barh(all_countries.index[::-1], all_countries.values[::-1], color=palette[::-1], edgecolor='white')

for bar in bars:
    ax.text(bar.get_width() + 15, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', ha='left', fontsize=9)

ax.set_title('Top 15 Countries by Number of Netflix Titles', pad=14)
ax.set_xlabel('Number of Titles')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EDA 6: Word Cloud of Descriptions (optional, skip if wordcloud not installed)
# ─────────────────────────────────────────────────────────────

try:
    from wordcloud import WordCloud
    text = ' '.join(df['description'].dropna().tolist())
    wc = WordCloud(width=900, height=450, background_color='black',
                   colormap='Reds', max_words=200, collocations=False)
    wc.generate(text)
    
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud – Netflix Descriptions', fontsize=16, fontweight='bold', color='white', pad=12)
    fig.patch.set_facecolor('black')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('ℹ️  wordcloud not installed – skipping word cloud.')
    print('   Install with: pip install wordcloud')

---
## ⚙️ Step 4: Feature Engineering – TF-IDF Vectorization

**What is TF-IDF?**  
- **TF (Term Frequency)**: How often a word appears in a document.  
- **IDF (Inverse Document Frequency)**: Penalises words that appear in many documents (e.g., *the*, *and*) — making rare, descriptive words more valuable.  
- The result is a **numerical vector** for each movie that captures its content profile.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Combine relevant text columns into a single 'soup' feature
# ─────────────────────────────────────────────────────────────

def create_feature_soup(row):
    """
    Combines title, genre, description, director, and cast into
    a single lowercase string (called 'soup') for TF-IDF input.
    
    Why lowercase? TF-IDF is case-sensitive by default, so 'Action'
    and 'action' would be treated as different words without lowercasing.
    """
    return ' '.join([
        str(row.get('title', '')),
        str(row.get('listed_in', '')),    # Genre
        str(row.get('description', '')),
        str(row.get('director', '')),
        str(row.get('cast', ''))
    ]).lower()

# Apply to every row → creates a new 'combined_features' column
df['combined_features'] = df.apply(create_feature_soup, axis=1)

# Preview the combined feature for the first 3 entries
print('─── Sample Combined Features ───')
for i in range(3):
    print(f'\nTitle: {df["title"][i]}')
    print(f'Soup : {df["combined_features"][i][:200]}...')

In [ ]:
# ─────────────────────────────────────────────────────────────
# TF-IDF Vectorization
# ─────────────────────────────────────────────────────────────

print('Building TF-IDF matrix...')

# TfidfVectorizer parameters:
#   stop_words='english' → Remove common English words ('the', 'and', 'is', etc.)
#   max_features=10000   → Use only the 10,000 most important words (faster, less memory)
#   ngram_range=(1,2)    → Use both single words and two-word pairs (bigrams)
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1, 2)    # Include bigrams like 'romantic comedy', 'true crime'
)

# fit_transform:
#   - fit: learns the vocabulary from all movie descriptions
#   - transform: converts each movie's soup into a TF-IDF vector
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print(f'✅  TF-IDF matrix built!')
print(f'   Shape: {tfidf_matrix.shape}')
print(f'   → {tfidf_matrix.shape[0]:,} movies × {tfidf_matrix.shape[1]:,} unique terms')
print(f'   Memory: {tfidf_matrix.data.nbytes / 1e6:.2f} MB (sparse representation)')

---
## 🤖 Step 5: Build the Recommendation System

**How Cosine Similarity Works:**  
Each movie is now a vector of TF-IDF scores. Cosine Similarity measures the **angle** between two vectors:  
- **Score = 1.0** → Identical content (same movie)  
- **Score → 0** → Completely different content  

We use `linear_kernel` instead of `cosine_similarity` — it's mathematically identical for normalized TF-IDF vectors but **much faster**.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Compute Cosine Similarity Matrix
# ─────────────────────────────────────────────────────────────

print('Computing cosine similarity matrix...')
print('(This compares every movie against every other movie — may take a moment)')

# linear_kernel computes dot product, equivalent to cosine similarity for L2-normalized vectors
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

print(f'✅  Cosine similarity matrix computed!')
print(f'   Shape: {cosine_sim.shape}  ({cosine_sim.shape[0]:,} × {cosine_sim.shape[1]:,})')
print(f'   Memory: {cosine_sim.nbytes / 1e6:.1f} MB')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Recommendation Function
# ─────────────────────────────────────────────────────────────

def recommend_movies(movie_name: str, top_n: int = 10) -> pd.DataFrame:
    """
    Recommend the top N most similar Netflix titles to the given movie.

    Algorithm:
      1. Find the index of the given movie in the dataframe.
      2. Extract the cosine similarity scores for that movie vs all others.
      3. Sort scores descending.
      4. Return the top_n entries (excluding the movie itself).

    Args:
        movie_name (str): Title of the movie to base recommendations on.
        top_n      (int): Number of recommendations to return (default: 10).

    Returns:
        pd.DataFrame with columns: rank, similarity_score, title, type,
                                   listed_in, rating, release_year
    """
    # ─ Step 1: Find the movie index ─────────────────────────────
    # Normalise to lowercase for flexible, case-insensitive matching
    movie_lower = movie_name.strip().lower()
    titles_lower = df['title'].str.lower()

    # Try exact match first
    exact = df[titles_lower == movie_lower]
    if not exact.empty:
        idx = exact.index[0]
        matched_title = df.loc[idx, 'title']
    else:
        # Fall back to partial/contains match
        partial = df[titles_lower.str.contains(movie_lower, na=False)]
        if partial.empty:
            print(f'❌  "{movie_name}" not found in the Netflix dataset.')
            print('   Tips:')
            print('   • Check for typos')
            print('   • Try a shorter/partial title (e.g. "Stranger" for "Stranger Things")')
            print('   • The title may not be in this dataset (8,807 titles)')
            return pd.DataFrame()
        idx = partial.index[0]
        matched_title = df.loc[idx, 'title']
        print(f'ℹ️  Exact match not found. Using: "{matched_title}"')

    print(f'\n🎬  Recommendations for: "{matched_title}"')
    print(f'    Type: {df.loc[idx, "type"]} | Genre: {df.loc[idx, "listed_in"]}\n')

    # ─ Step 2: Get similarity scores ────────────────────────────
    # cosine_sim[idx] gives the similarity of this movie to ALL others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # ─ Step 3: Sort by similarity (descending) ──────────────────
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # ─ Step 4: Exclude the input movie itself (score = 1.0) ──────
    sim_scores = sim_scores[1: top_n + 1]

    # ─ Step 5: Build result dataframe ───────────────────────────
    indices  = [i[0] for i in sim_scores]
    scores   = [round(i[1], 4) for i in sim_scores]

    results = df.iloc[indices][['title', 'type', 'listed_in', 'rating', 'release_year']].copy()
    results.insert(0, 'similarity_score', scores)
    results.index = range(1, len(results) + 1)  # Rank from 1
    results.index.name = 'rank'

    return results

print('✅  recommend_movies() function defined and ready!')

---
## 🎯 Step 6: Demo – Get Movie Recommendations

Let's test the recommendation system with **three sample titles**.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo 1: Recommendations for 'Inception'
# ─────────────────────────────────────────────────────────────

print('=' * 65)
results_1 = recommend_movies('Inception', top_n=10)
if not results_1.empty:
    display(results_1)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo 2: Recommendations for 'The Crown'
# ─────────────────────────────────────────────────────────────

print('=' * 65)
results_2 = recommend_movies('The Crown', top_n=10)
if not results_2.empty:
    display(results_2)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo 3: Recommendations for 'Money Heist'
# ─────────────────────────────────────────────────────────────

print('=' * 65)
results_3 = recommend_movies('Money Heist', top_n=10)
if not results_3.empty:
    display(results_3)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Visualise Recommendations: Similarity Scores as Bar Chart
# ─────────────────────────────────────────────────────────────

def plot_recommendations(results: pd.DataFrame, query_title: str):
    """Plot horizontal bar chart of recommendation similarity scores."""
    if results.empty:
        return
    
    fig, ax = plt.subplots(figsize=(11, 5))
    palette = sns.color_palette('coolwarm_r', len(results))
    
    bars = ax.barh(
        results['title'][::-1],
        results['similarity_score'][::-1],
        color=palette[::-1],
        edgecolor='white',
        linewidth=0.8
    )
    
    for bar in bars:
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.3f}', va='center', ha='left', fontsize=9)
    
    ax.set_xlim(0, results['similarity_score'].max() * 1.18)
    ax.set_title(f'Top {len(results)} Recommendations for "{query_title}"\n(Cosine Similarity Score)', pad=12)
    ax.set_xlabel('Cosine Similarity Score')
    ax.set_ylabel('Recommended Title')
    plt.tight_layout()
    plt.show()

# Plot for all three demo queries
if not results_1.empty: plot_recommendations(results_1, 'Inception')
if not results_2.empty: plot_recommendations(results_2, 'The Crown')
if not results_3.empty: plot_recommendations(results_3, 'Money Heist')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Error Handling Demo
# ─────────────────────────────────────────────────────────────
# Test that non-existent titles give a helpful error message
# and don't crash the program.

print('─── Testing error handling ───')
result_not_found = recommend_movies('This Title Does Not Exist 12345', top_n=5)
print('\nFunction returned gracefully without crashing ✅')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bonus: Interactive Search
# ─────────────────────────────────────────────────────────────
# Change the variable below to try your own movie title!

MY_MOVIE = 'Breaking Bad'   # ← Change this to any Netflix title
MY_TOP_N = 10               # ← Number of recommendations

print('=' * 65)
my_results = recommend_movies(MY_MOVIE, top_n=MY_TOP_N)
if not my_results.empty:
    display(my_results)
    plot_recommendations(my_results, MY_MOVIE)

---
## 🏁 Summary

| Step | Description | Status |
|------|-------------|--------|
| 1 | Dataset Loading | ✅ |
| 2 | Data Cleaning (duplicates, NaN, whitespace) | ✅ |
| 3 | EDA (genre, year, ratings, countries) | ✅ |
| 4 | Feature Engineering (TF-IDF on combined text) | ✅ |
| 5 | Cosine Similarity Matrix | ✅ |
| 6 | `recommend_movies()` with error handling | ✅ |
| 7 | Demo for Inception, The Crown, Money Heist | ✅ |

---

**Next Steps to Improve:**
- Try **Collaborative Filtering** (based on user watch history rather than content)
- Add **weight multipliers** (e.g., title word × 3, description × 1) to tune relevance
- Add a **web interface** using Streamlit or Flask
- Include **IMDb ratings** for score-based hybrid recommendations